# Lecture 27 - SciPy I: a rate, and how sure you are
### EES 3350/5350 · Python in Earth Science · Monday, November 9, 2026 

In Lecture 24, `np.polyfit` told us CO$_2$ at Mauna Loa rose about **1.63 ppm per year**. In Lab 10, you're computing rates of sea-level change the same way.

But a number by itself isn't a scientific result. If a classmate fit slightly different years and got 1.58, would that **disagree** with you? You can't tell without knowing **how sure** each number is. Scientists write a rate ith its uncertainty: **1.63 ± 0.06 ppm/yr**.

Today: **SciPy**, Python's big toolbox for science, and its function `linregress`, which gives you the slope **and** its uncertainty.

**Learning objectives**
- explain what SciPy is and how it relates to NumPy
- fit a line with `stats.linregress` and pull out the slope, intercept, $r$, $p$, and standard error
- report a rate as a **slope ± uncertainty**
- show uncertaintites on a figure with error bars
- use error bars to decide whether two rates **really** differ
- explain what makes a slope more or less certain, and why a tiny $p$-value doesn't eman a line is the right model



--
### Part 0 : set up

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

co2 = pd.read_csv("../../course-materials/week10/co2_maunaloa_annual.csv", header=None, names=["year", "ppm"])

co2.tail(3)

<span style="font-size:0.85em;color:#6b7a8d;"><em>Data: annual mean CO₂ at Mauna Loa Observatory, 1959–2022. Dr. Xin Lan, NOAA Global Monitoring Laboratory (gml.noaa.gov/ccgg/trends/) and Dr. Ralph Keeling, Scripps Institution of Oceanography (scrippsco2.ucsd.edu/).</em></span>

---
## Part 1 : what `np.polyfit` gave us (review)

In [ ]:
coeffs = np.polyfit(co2["year"], co2["ppm"], 1)

print(f"slope: {coeffs[0]:.2f} ppm per year")

`np.polyfit` gave us back two numbers, the slope and the intercept. 
<center><b>What didn't it tell us?</b></center>


<details>
<summary style="color:green;"><strong>Answer</strong></summary>

How **certain** that slope is. Is it 1.63 give or take 0.01? Give or take 1? `np.polyfit` doesn't say, so you can't tell whether a different number from a classmate (or another study) really diagrees with yours.

</details>

---
## Part 2  : SciPy and `linregress`

**SciPy** ("Scientific Python") is a big library of scientific tools, built on top of NumPy. It's organized into **submodules**, one per kind of job:
|**submodule**|**for ...**|**when in this class**|
|:--|:--|:--|
|`scipy.stats`|statistics: fits, correlations, tests|today and Friday|
|`scipy.interpolate`|filling in values between measurements|Friday|
|`scipy.signal`|finding peaks in a noisy record|Friday|
|`scipy.optimize`|fitting curves that aren't straight lines|Friday|
|`scipy.integrate`|totals: the area under a curve|next Monday|

Today is a close look at **one** tool. On Friday, we'll step back and tour the whole toolbox: ohw to find, read, and test **any** SciPy function, even ones we never cover in class.

You import just the submodule you need:

In [ ]:
from scipy import stats

`stats.linregress(x, y)` fits the same least-squares line as `np.polyfit(x, y, 1)` ("Linear regression" is the statistics name for fitting a line.)

In [ ]:
result = stats.linregress(co2["year"], co2["ppm"])

result

#### What just happened?

`linregress` gave back **one** object, `result`, that holds five numbers. Pull each one out with a dot and its name, just like `.shape` on an array:

In [ ]:
print("slope:          ", result.slope)
print("intercept:      ", result.intercept)
print("r value:        ", result.rvalue)
print("p value:        ", result.pvalue)
print("standard error: ", result.stderr)

|**name**|**what it is**|**here**|
|:--|:--|:--|
|`slope`| the rate: how much $y$ changes when $x$ goes up by 1| ppm per year, **the same as `np.polyfit`**|
|`intercept`| $y$ when $x$ is 0| (meaningless for year 0, Lecture 24)|
|`rvalue`|$r$, the **correlation**: how close the points are to a line, from -1 to 1| very close to 1|
|`pvalue`|the chance of a slope this big if there were **really no trend**|tiny|
|`stderr`|the **standard error** of the slope: how uncertain the slope is|the new, important one|

`r` squared, `r²`, is the fraction of the ups and downs in $y$ that the line explains: `result.rvalue**2`

<div class="alert alert-success">

**Tip**: standard error ≠ standard deviation
- **standard deviation** (SD) describes the **data**: how spread out the individual points are. For CO$_2$, the points scatter about **4.1 ppm** around the straight line. That's the residual from Lecture 24)
- **standard error** (SE) describes an **estimate made from the data**: like a slope or a mean: how much that estimate would wobble if you could collect data again. The CO$_2$ slope's SE is **0.028 ppm/year**.

The key difference is what happens when you collect **more data**:
- The **SD stays about the same.** More measurments don't make the world less scattered.
- The **SE shrinks.** More measurements pin the estimate down better.
    
</div>

---
## Part 3 : slope ± uncertainty

<center>What does the <b>standard error</b> mean?</center>

Imagine you could re-run history many times, each time with slightly different weather and slightly different measurements. You'd get a slightly different slope each time. The standard error is about how much those slopes would **spread out**.

The common way to report a rate is **slope ± 2 standard errors**. That range is roughly a **95% confidence interval**: we're about 95% sure the true rate is inside it.

In [ ]:
slope = result.slope
uncertainty = 2 * result.stderr

print(f"CO2 rose {slope:.2f} ± {uncertainty:.2f} ppm per year, 1959-2022")

The `±` symbol can go right into an f-string. We rounded uncertainty to the same number of decimal places as the slope, because more digits than that would claim a precision we don't have.

### Figure 1 : the data and the line
`linregress` doesn't have a `polyval`, but you don't need one: a line is just `slope * x + intercept`.

In [ ]:
fitted = result.slope * co2["year"] + result.intercept

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(co2["year"], co2["ppm"], "o", markersize=3, color="gray", label="measured")
ax.plot(co2["year"], fitted, color="tab:olive", label=f"{slope:.2f} ± {uncertainty:.2f} ppm/yr")

ax.set_xlabel("year")
ax.set_ylabel("CO$_2$ (ppm)")
ax.legend();

---
## Part 4 : do two rates really differ?

In Lecture 24, the rate depended on **which years** you fit. Now wec an ask whether those differences are **real**, or just he kind of wobble the uncertainty allows.

<div class="alert alert-info">

### Exercise 1 : the last 10 years

1. Keep only 2013-2022.
2. Fit a line with `linregress`.
3. print the rate ± 2 standard erros.
    
</div>

In [ ]:
recent = co2[co2["year"] >= ____]

recent_result = stats.linregress(recent["____"], recent["____"])

rcent_slope = recent_result.____
recent_uncertainty = 2 * recent_result.____

print(f"2013-2022: {recent_slope:.2f} ± {recent_uncertainty:.2f} ppm per year")

<div class="alert alert-warning">

### Question 1

Write each rate as a range: the full record is (1.63- 0.06) to (1.63 + 0.06) so **1.57 to 1.69**. 

**What's the range for the last 10 years of the record?** Do the two ranges overlap? What does that tell you?
    
</div>

> your answer here

<details>
<summary style="color:green;"><strong>Answer</strong></summary>

The last 10 years is **2.41 to 2.55** ppm/yr. The two ranges don't come close to overlapping...so the difference is **real**: CO$_2$ is rising about 1.5 times faster now than the 1959-2022 average. Without the `±` numbers, you couldn't have said that with confidence.

If two ranges **do** overlap, you can't claim the rates are different (in Lab 10, that's how to judge whether your station's rate really differs from Honolulu's).

</details>

<div class="alert alert-info">

### Exercise 2 : rate vs window length

Fit the **most recent** 5, 10, 20, 40, and 64 yeas, and plot each rate with it's uncertainty.
    
</div>

In [ ]:
windows = [5, 10, 20, 40, 65]
slopes = []
uncertainties = []

for n_years in windows:
    piece = co2.tail(____)
    r = stats.linregress(piece["year"], piece["ppm"])
    slopes.append(r.____)
    uncertainties.append(2*r.____)

fig, ax = plt.subplots(figsize=(6, 4))
ax.errorbar(windows, slopes, yerr=uncertainties, fmt="o", capsize=5, color="black")

ax.set_xlabel("years of dat in the fit (ending in 2022)")
ax.set_ylabel("rate of CO$_2$ rise (ppm/year)")
ax.set_xticks(windows);

<details>
<summary style="color:green;"><strong>Things to see in this plot</strong></summary>

There are two things to see in this plot:
- **The rates climb** the more recent the window: 1.63 for all 64 years, 1.92 for 40, 2.27 for 20, about 2.5 for the last 10 to 5. The bars don't overlap between the long and short windows, so the speed-up is real.
- **The 5-year bar is by far the biggest** (± 0.20): with only 5 points, one unusual year tilts the line a lot. **Fewer points, less certain**. But notice the 64-year bar is **not** the smallest, even with the msot points! That's because a straight line fits 64 years of **curving** data badly, and a bad fit makes the slope less certain too. Two things set the ±: **how many points** and **how well line fits**.

`ax.errorbar` is new: like `ax.plot`, but 
- `yerr=` draws a bar above and below each point
- `fmt="o"` draws dots with no line
- `capsize=` adds little caps at the ends

<div class="alert alert-danger">

**Common Error**: a tiny $p$-value and an $r$ near 1 do **not** mean a straight line is the right model. Remember the **U-shaped residuals** from Lecture 24? CO$_2$ curves upward so the full=-record line is wrong at the ends. The standard error also *8assumes** the line is the right shape and the leftovers are random. When residuals have a pattern, the ± is **too small**. Always plot the residuals before you trust the ±.
    
</div>

---
## Extra, if time allows:
### **Why "2"?** the normal and t distributions

Imagine you could re-run history many times and fit a slope each time. those slopes would pile up in a bell curve (a **normal distribution**) centered on the true slope. The **standard error** is the width of htat bell curve.

For any normal distribution:
- about **68 %** of values fall within **±1 SE**
- about **95 %** fall within **±1.96 SE**, which people round to 2

The bell-curve math assumes we know SE *exactly*, but we don't. `linregress` only **estimates** the standard error, from the scatter of our data. 
- with lots of points, that estimate is solid.
- with few points, the estimate is shaky.

So we need a wider range to be 95% sure. The **t distribution** is a bell curve with **fatter tails** that accounts for this.
- the fewer the points you have, the fatter the tails, and the bigger the multiplyer you need to reach 95%.
    - the degrees of freedom are teh number of points minus the 2 numbers the line estimated (slope and intercept), so `n-2`

In [ ]:
x = np.linspace(-5, 5, 500)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(x, stats.norm.pdf(x), color="black", linewidth=2, label="normal (infinity many points)")

number_of_points = [64, 10, 5]
color = ["#0072B2", "#E69F00", "#D55E00"]

for n, color in zip(number_of_points, color):
    ax.plot(x, stats.t.pdf(x, n-2), color=color, label=f"t distribution, {n} points")
    cutoff = stats.t.ppf(0.975, n-2)
    ax.axvline(cutoff, color=color, linestyle="--", linewidth=1)
    ax.axvline(-cutoff, color=color, linestyle="--", linewidth=1)

ax.axvline(1.96, color="black", linestyle="--", linewidth=1)
ax.axvline(-1.96, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("distance from the true slope (instandard errors)")
ax.set_ylabel("how likely")
ax.set_title("Dashed lines: the edges of the middle 95%")
ax.legend(fontsize=8);

#### What just happened?

- With 64 points, the t curve sits right on top of the normal curve, and its 95% edges are at ±2.00.
- With 10 points, the tails are fatter and the edges move about to about ±2.31
- With only 5 points, they're all the way out at ±3.18

`stats.t.ppf(0.975, n-2)` gives that edge: `0.975` leaves 2.5% in each tail, so 95% sits in the middle.

In [ ]:
windows = [5, 10, 20, 64]
slopes = []
shortcut = []
exact = []

for n_years in windows:
    piece = co2.tail(n_years)
    r = stats.linregress(piece["year"], piece["ppm"])
    multiplier = stats.t.ppf(0.975, n_years - 2)

    slopes.append(r.slope)
    shortcut.append(2 * r.stderr)
    exact.append(multiplier * r.stderr)
    print(f"{n_years:2d} years: multiplier {multiplier:.2f}   shortcut ± {2 * r.stderr:.2f}   exact ± {multiplier * r.stderr:.2f}")

positions = np.arange(len(windows))

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(positions - 0.1, slopes, yerr=shortcut, fmt="o", capsize=5, color="gray", label="± 2 SE (shortcut)")
ax.errorbar(positions + 0.1, slopes, yerr=exact, fmt="o", capsize=5, color="black", label="± t × SE (exact 95%)")

ax.set_xticks(positions)
ax.set_xticklabels([f"last {n} years" for n in windows])
ax.set_ylabel("rate of CO$_2$ rise (ppm/yr)")
ax.legend();

#### What just happened?

For 20 or more poitns, the shortcut and the exact answer are the same. For 5 points, the honest 95% range is about **half again as wide** as "± 2 SE" **The fewer points you have, the more the shortcut overstates how sure you are.**

In the figure above, each pair if bars is offset by ±0.1 so they sit side by side isntead of on top of each other.

<div class="alert alert-success">

**Tip: what you'll see in papers, and what to do yourself**

- **Always say what your ± means** (± 1 standard error, ± 2 standard errors, or a 95% confidence interval). Papers use all three, and readers can't compare numbers without knowing which. This one is expected everywhere.
- **Always say which years (or which stretch of river) a rate covers.** Also expected everywhere.
- Most papers use ± 1 or ± 2 standard errors straight from the fit. That's fine with lots of data. With **fewer than about 30 points**, the t multiplier is the more honest choice.
- Plot the residuals before trusting any ±. Few papers show this step, but careful scientists always do it.

</div>